# Notebook 01 — Common Table Expressions (CTE)

Tema 08. Una **CTE** (*Common Table Expression*, cláusula `WITH`) es una subconsulta con nombre que se declara al inicio de la query y se usa como si fuera una tabla. Dos usos: **legibilidad** (descomponer una query compleja en pasos con nombre) y **recursión** (`WITH RECURSIVE`) para recorrer jerarquías.

Cubres: `UNION ALL` / `UNION` (apilar resultados, con y sin duplicados) — la pieza que usan las CTE recursivas —, `WITH` simple y encadenadas, y `WITH RECURSIVE` (caso base + caso recursivo + `UNION ALL`) para el análisis jerárquico empleado→jefe.

**Contenido de este notebook:**

- [Setup](#setup)
- [`UNION ALL`](#union-all)
- [`WITH` — una subconsulta con nombre](#with--una-subconsulta-con-nombre)
- [Múltiples CTE encadenadas](#múltiples-cte-encadenadas)
- [`WITH RECURSIVE` — jerarquías](#with-recursive--jerarquías)
- [Recorrido ascendente: la cadena de mando](#recorrido-ascendente-la-cadena-de-mando)

## Setup

In [ ]:
# Setup — instala JupySQL si hace falta (Colab trae ipython-sql, no JupySQL).
import importlib.util, subprocess, sys
if importlib.util.find_spec("jupysql") is None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "ipython-sql"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupysql"], check=True)
    print("⚠ JupySQL instalado. REINICIA el kernel (Entorno de ejecución → Reiniciar sesión)")
    print("  y vuelve a correr esta celda y las siguientes.")
else:
    print("✓ JupySQL listo.")

In [ ]:
%load_ext sql

from sqlalchemy import create_engine

# Reemplaza con tus valores del Tema 01
AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

%sql engine
# Devuelve cada query como DataFrame de pandas (mejor render en Colab, y
# el resultado es directamente manipulable con pandas).
%config SqlMagic.autopandas = True

## `UNION ALL`

`UNION ALL` apila los resultados de dos o más `SELECT` (mismas columnas) **sin eliminar duplicados**. Es la pieza que arma las CTE recursivas que verás más abajo (pega el caso base con el caso recursivo), y por sí solo sirve para combinar conteos o *facts* de distinto grano:

In [ ]:
%%sql
SELECT 'producto' AS entidad, COUNT(*) AS n FROM northwind_dwh.dim_product
UNION ALL
SELECT 'cliente',             COUNT(*)      FROM northwind_dwh.dim_customer
UNION ALL
SELECT 'empleado',            COUNT(*)      FROM northwind_dwh.dim_employee;

`UNION ALL` no compara ni ordena: solo concatena, por eso es más rápido que `UNION` (que **sí** elimina duplicados, con su costo de ordenamiento). En CTE recursivas siempre se usa `UNION ALL`.

### `UNION` vs `UNION ALL` — la diferencia con datos

Para ver el efecto necesitamos filas **repetidas**. Tomemos las ciudades del Reino Unido donde hay **clientes** o **empleados**: `London` aparece muchas veces (6 clientes y 4 empleados ahí). Primero con `UNION ALL`, que apila todo tal cual:

In [ ]:
%%sql
SELECT city FROM northwind_oltp.customers WHERE country = 'UK'
UNION ALL
SELECT city FROM northwind_oltp.employees WHERE country = 'UK'
ORDER BY city;

**11 filas** — `London` aparece 10 veces (6 + 4) y `Cowes` una. Ahora exactamente la misma query, pero con `UNION` (sin `ALL`):

In [ ]:
%%sql
SELECT city FROM northwind_oltp.customers WHERE country = 'UK'
UNION
SELECT city FROM northwind_oltp.employees WHERE country = 'UK'
ORDER BY city;

Solo **2 filas**: `UNION` colapsó las 10 `London` en una y dejó `Cowes`. Lo único que cambió es la palabra `ALL`:

| | Qué hace | Filas | Costo |
|---|---|---|---|
| `UNION ALL` | concatena, **conserva** cada fila | 11 | barato (solo apila) |
| `UNION` | **elimina duplicados** | 2 | más caro (ordena/compara para deduplicar) |

**Cuándo cuál:**

- **`UNION ALL`** por defecto — cuando no hay duplicados, o cuando sí los hay pero los quieres conservar (p. ej. apilar *facts* de distinto grano). Es el que usan las CTE recursivas.
- **`UNION`** solo cuando **necesitas** la lista de valores únicos. Internamente equivale a un `UNION ALL` seguido de un `DISTINCT`, con ese costo extra de ordenamiento.

## `WITH` — una subconsulta con nombre

Sintaxis: `WITH nombre AS ( subconsulta ) SELECT … FROM nombre`. La CTE se calcula y se referencia como una tabla temporal que solo vive en esa query. Convierte una query anidada en **pasos lineales legibles**:

In [ ]:
%%sql
WITH ventas_categoria AS (
    SELECT dp.category_name,
           ROUND(SUM(fs.line_total), 2) AS ventas
    FROM   northwind_dwh.fact_sales fs
    JOIN   northwind_dwh.dim_product dp ON dp.product_key = fs.product_key
    GROUP  BY dp.category_name
)
SELECT category_name, ventas
FROM   ventas_categoria
WHERE  ventas > 100000
ORDER  BY ventas DESC;

La CTE `ventas_categoria` se define una vez y se usa en el `SELECT` externo. Sin CTE, tendrías que meter ese `SUM`/`GROUP BY` como subquery anidada dentro del `FROM` — funciona, pero se lee de adentro hacia afuera. La CTE lo hace de arriba hacia abajo.

## Múltiples CTE encadenadas

Puedes declarar varias CTE separadas por coma; cada una puede referirse a las anteriores. Se leen como una secuencia de pasos:

In [ ]:
%%sql
WITH ventas_producto AS (
    SELECT dp.category_name, dp.product_name,
           ROUND(SUM(fs.line_total), 2) AS ventas
    FROM   northwind_dwh.fact_sales fs
    JOIN   northwind_dwh.dim_product dp ON dp.product_key = fs.product_key
    GROUP  BY dp.category_name, dp.product_name
),
ranking AS (
    SELECT category_name, product_name, ventas,
           ROW_NUMBER() OVER (PARTITION BY category_name ORDER BY ventas DESC) AS rn
    FROM   ventas_producto
)
SELECT category_name, product_name, ventas
FROM   ranking
WHERE  rn = 1
ORDER  BY ventas DESC;

`ranking` consume `ventas_producto`. El resultado: el producto más vendido de cada categoría — el mismo patrón **top-N por grupo** del Tema 07, ahora expresado como pasos con nombre.

## `WITH RECURSIVE` — jerarquías

Una CTE recursiva tiene tres partes:

1. **Caso base** (ancla): la fila o filas de arranque.
2. **`UNION ALL`**.
3. **Caso recursivo**: un `SELECT` que se refiere **a la propia CTE**, con una **condición de parada** que evita que corra para siempre.

PostgreSQL ejecuta el caso base una vez y luego repite el caso recursivo, alimentándose de las filas recién producidas, hasta que no genera ninguna nueva.

El ejemplo mínimo para ver el mecanismo no necesita ninguna tabla: **generar una serie de fechas** entre dos límites. El caso base es el primer día; el caso recursivo suma un día hasta llegar al final.

In [ ]:
%%sql
WITH RECURSIVE calendario AS (
    -- caso base: el primer día del periodo
    SELECT DATE '1997-01-01' AS dia
    UNION ALL
    -- caso recursivo: suma un día mientras no se llegue al final
    SELECT dia + 1
    FROM   calendario
    WHERE  dia < DATE '1997-01-15'
)
SELECT dia,
       EXTRACT(DOW FROM dia) AS dia_semana   -- 0 = domingo … 6 = sábado
FROM   calendario;

Quince filas a partir de una sola: el caso base produce `1997-01-01` y el caso recursivo va sumando un día (`dia + 1`) mientras se cumpla `dia < '1997-01-15'`; cuando deja de cumplirse, la recursión para. Generar fechas así —**sin** una tabla de calendario— es muy útil para **rellenar huecos** en series temporales (p. ej. días sin ventas que un `GROUP BY` omitiría por no tener filas).

El mismo patrón sirve para algo más estructural: recorrer una **jerarquía** empleado→jefe.

> Usamos `northwind_oltp.employees` (no el DWH): tiene la columna `reports_to` (el `employee_id` del jefe). El `dim_employee` del DWH guarda la jerarquía aplanada (`reports_to_name`), sin la auto-referencia que la recursión necesita.

In [ ]:
%%sql
WITH RECURSIVE jerarquia AS (
    -- caso base: el empleado raíz (sin jefe)
    SELECT employee_id,
           first_name || ' ' || last_name AS nombre,
           reports_to,
           1 AS nivel
    FROM   northwind_oltp.employees
    WHERE  reports_to IS NULL

    UNION ALL

    -- caso recursivo: empleados cuyo jefe ya está en la jerarquía
    SELECT e.employee_id,
           e.first_name || ' ' || e.last_name,
           e.reports_to,
           j.nivel + 1
    FROM   northwind_oltp.employees e
    JOIN   jerarquia j ON e.reports_to = j.employee_id
)
SELECT nivel,
       nombre,
       employee_id,
       reports_to
FROM   jerarquia
ORDER  BY nivel, employee_id;

El caso base ancla en el jefe sin superior (`reports_to IS NULL`). El caso recursivo une cada empleado con su jefe **ya presente** en `jerarquia`, sumando 1 al `nivel`, que indica la profundidad de cada empleado en la jerarquía. Sin recursión necesitarías un `JOIN` por cada nivel y tendrías que saber de antemano cuántos hay.

## Recorrido ascendente: la cadena de mando

La misma maquinaria, al revés: partir de un empleado y subir hasta la cima. Solo cambia la condición del `JOIN` (`e.employee_id = c.reports_to`, en vez de lo contrario):

In [ ]:
%%sql
WITH RECURSIVE cadena AS (
    -- caso base: el empleado de interés
    SELECT employee_id, first_name || ' ' || last_name AS nombre, reports_to, 1 AS paso
    FROM   northwind_oltp.employees
    WHERE  employee_id = 9          -- Anne Dodsworth

    UNION ALL

    -- caso recursivo: subir al jefe del que ya tenemos
    SELECT e.employee_id, e.first_name || ' ' || e.last_name, e.reports_to, c.paso + 1
    FROM   northwind_oltp.employees e
    JOIN   cadena c ON e.employee_id = c.reports_to
)
SELECT paso, nombre, employee_id, reports_to
FROM   cadena
ORDER  BY paso;

Resultado: Anne Dodsworth → Steven Buchanan → Andrew Fuller. La recursión se detiene cuando llega al empleado con `reports_to` nulo (no hay jefe al que subir).

## Cierre

Lo que cubriste:

| Concepto | Sintaxis |
|---|---|
| CTE simple | `WITH nombre AS (…) SELECT … FROM nombre` |
| CTE encadenadas | `WITH a AS (…), b AS (… FROM a) …` |
| Recursión | `WITH RECURSIVE c AS (base UNION ALL recursivo)` |
| Jerarquía descendente | caso base = raíz; recursivo baja niveles |
| Jerarquía ascendente | caso base = hoja; recursivo sube al jefe |
| Apilar resultados | `UNION ALL` (sin deduplicar) vs `UNION` (deduplica) |

---

<p align="center">
<a href="../Tema-07/Readme.md">← Anterior: Tema 07</a> | <a href="Readme.md">Volver al índice</a> | <a href="../Tema-09/Readme.md">Siguiente: Tema 09 →</a>
</p>